# Emotion Detection — MobileNetV2 on FER-2013
**COS30082 — Hasindu**

Driver notebook. All real logic lives in `emotion/src/` (importable Python modules). This notebook just orchestrates and produces the plots / metrics that go into the group report.

**Pipeline:**
1. Mount Drive + clone repo + install deps
2. Download FER-2013 from Kaggle into Drive (once)
3. EDA — class counts, sample grid
4. Build DataLoaders
5. Build model (MobileNetV2, ImageNet pretrained, last 4 blocks unfrozen, new 7-class head)
6. Train with class-weighted cross-entropy, checkpoint every epoch to Drive
7. Evaluate on FER-2013 public test set
8. Smoke-test the `EmotionPredictor` (the only thing the UI teammate needs)

## 1. Setup: Drive, repo, dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo into Colab's ephemeral disk and check out emotion-branch.
# If you already cloned earlier in this session, this cell is safe to re-run.
%cd /content
![ -d COS30082-Facial-Recognition ] || git clone https://github.com/jeromeliao03/COS30082-Facial-Recognition.git
%cd /content/COS30082-Facial-Recognition
!git fetch origin && git checkout emotion-branch && git pull --ff-only

In [ ]:
!pip install -q -r emotion/requirements.txt

In [ ]:
# Make `from emotion.src...` work from the notebook by adding repo root to path.
import sys, os
REPO_ROOT = '/content/COS30082-Facial-Recognition'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Download FER-2013 from Kaggle into Drive (run once)

In [ ]:
# Get kaggle.json into ~/.kaggle. Tries, in order:
#   1. ~/.kaggle/kaggle.json already there (re-run case)
#   2. Drive at /content/drive/MyDrive/MLGroup/kaggle.json
#   3. Prompt for upload from laptop (and persist to Drive for next time)
import os, shutil, pathlib
from emotion.src import config as C

DATA_DIR = pathlib.Path(C.DATA_DIR)
DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
C.MODELS_DIR.mkdir(parents=True, exist_ok=True)
C.REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def _ensure_kaggle_json():
    dst = '/root/.kaggle/kaggle.json'
    if os.path.exists(dst):
        return
    os.makedirs('/root/.kaggle', exist_ok=True)
    drive_path = '/content/drive/MyDrive/MLGroup/kaggle.json'
    if os.path.exists(drive_path):
        shutil.copy(drive_path, dst)
    else:
        print('kaggle.json not found on Drive. Upload it from your laptop now...')
        from google.colab import files
        uploaded = files.upload()  # pick kaggle.json
        assert 'kaggle.json' in uploaded, 'Please upload a file literally named kaggle.json'
        with open(dst, 'wb') as f:
            f.write(uploaded['kaggle.json'])
        # Persist to Drive so future Colab sessions skip the upload.
        os.makedirs('/content/drive/MyDrive/MLGroup', exist_ok=True)
        shutil.copy(dst, drive_path)
        print('Saved kaggle.json to Drive:', drive_path)
    os.chmod(dst, 0o600)

if not (DATA_DIR / 'train').exists():
    _ensure_kaggle_json()
    !kaggle datasets download -d msambare/fer2013 -p /content --unzip
    # The Kaggle archive expands to /content/train and /content/test.
    shutil.move('/content/train', str(DATA_DIR / 'train'))
    shutil.move('/content/test',  str(DATA_DIR / 'test'))
    print('Dataset extracted to', DATA_DIR)
else:
    print('Dataset already on Drive:', DATA_DIR)

print('train classes:', sorted(os.listdir(DATA_DIR / 'train')))
print('test  classes:', sorted(os.listdir(DATA_DIR / 'test')))

## 3. EDA — class distribution + sample images

In [ ]:
import os, random
import matplotlib.pyplot as plt
from PIL import Image
from emotion.src import config as C

counts = {cls: len(os.listdir(C.TRAIN_DIR / cls)) for cls in C.CLASSES}
print(counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(counts.keys()), list(counts.values()))
ax.set_title('FER-2013 train: images per class')
ax.set_ylabel('count')
fig.tight_layout(); plt.show()
fig.savefig(C.REPORTS_DIR / 'class_distribution.png', dpi=150)

In [ ]:
# Sample grid: one row per class, 5 random examples each.
random.seed(0)
fig, axes = plt.subplots(len(C.CLASSES), 5, figsize=(10, 12))
for r, cls in enumerate(C.CLASSES):
    files = os.listdir(C.TRAIN_DIR / cls)
    picks = random.sample(files, 5)
    for c, fn in enumerate(picks):
        img = Image.open(C.TRAIN_DIR / cls / fn)
        axes[r, c].imshow(img, cmap='gray')
        axes[r, c].axis('off')
        if c == 0:
            axes[r, c].set_ylabel(cls, rotation=0, ha='right', va='center')
fig.suptitle('FER-2013 samples'); fig.tight_layout(); plt.show()
fig.savefig(C.REPORTS_DIR / 'sample_grid.png', dpi=150)

**Observation for the report:** `disgust` has ~547 images vs. ~7000 for `happy` — a ~13× imbalance. Mitigated by passing inverse-frequency class weights into `CrossEntropyLoss` (see `data.py` and `train.py`).

## 4. DataLoaders

In [ ]:
from emotion.src.data import build_dataloaders
train_loader, val_loader, test_loader, class_weights, train_counts = build_dataloaders()
print('train batches:', len(train_loader), '| val batches:', len(val_loader),
      '| test batches:', len(test_loader))
print('class weights:', dict(zip(C.CLASSES, [round(w, 3) for w in class_weights.tolist()])))

# Sanity-check one batch shape.
xb, yb = next(iter(train_loader))
print('batch x:', xb.shape, xb.dtype, '| y:', yb.shape, yb.dtype)

## 5. Model

In [ ]:
from emotion.src.model import build_mobilenetv2, count_trainable
model = build_mobilenetv2()
trainable, total = count_trainable(model)
print(f'Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

## 6. Train

In [ ]:
from emotion.src.train import train
history = train(
    model, train_loader, val_loader, class_weights,
    epochs=C.EPOCHS, tag='mobilenetv2',
)
print('Best val acc:', history.best_val_acc, 'at epoch', history.best_epoch)

In [ ]:
from dataclasses import asdict
from emotion.src.evaluate import plot_training_curves
fig = plot_training_curves(asdict(history),
                           save_path=C.REPORTS_DIR / 'training_curves.png')
plt.show()

## 7. Evaluate on the public test set

In [ ]:
from emotion.src.model import load_checkpoint
from emotion.src.evaluate import collect_predictions, metrics_report, plot_confusion_matrix

best = load_checkpoint(C.MODELS_DIR / 'mobilenetv2_best.pth')
y_true, y_pred, y_prob = collect_predictions(best, test_loader)
report = metrics_report(y_true, y_pred)
print('Test accuracy   :', round(report['accuracy'],   4))
print('Test macro-F1   :', round(report['macro_f1'],   4))
print('Test weighted-F1:', round(report['weighted_f1'],4))
print()
print(report['classification_report'])

In [ ]:
fig = plot_confusion_matrix(report['confusion_matrix'],
                            save_path=C.REPORTS_DIR / 'confusion_matrix.png',
                            normalize=True)
plt.show()

# Also dump raw numbers as JSON for the report.
import json
with open(C.REPORTS_DIR / 'test_metrics.json', 'w') as f:
    json.dump({k: v for k, v in report.items() if k != 'classification_report'}, f, indent=2)

**Discussion bullets for the report:**
- Expect ~65–70% test accuracy on FER-2013 with this recipe; SOTA is ~75% so this is a competitive baseline.
- Look at the confusion matrix for the two famous confusion pairs: **fear↔surprise** and **sad↔neutral**. Both are linguistically and visually overlapping in FER-2013.
- Per-class F1 for `disgust` is the key metric — without class weights it tends to ~0; with them you should see a meaningful number (the report can cite both ablations if time permits).

## 8. Smoke-test the predictor (the UI's entrypoint)

In [ ]:
from emotion.src.predict import EmotionPredictor
import numpy as np
from PIL import Image

predictor = EmotionPredictor(C.MODELS_DIR / 'mobilenetv2_best.pth', input_is_bgr=False)

# Grab one test image, run it through the public API, print result.
sample_dir = C.TEST_DIR / 'happy'
sample_path = sample_dir / sorted(os.listdir(sample_dir))[0]
img = Image.open(sample_path)
result = predictor.predict(np.array(img.convert('RGB')))
print('file       :', sample_path.name)
print('true class : happy')
print('prediction :', result)

## 9. Hand-off checklist
- `models/mobilenetv2_best.pth` — give to UI teammate (or share via Drive).
- `emotion/src/predict.py` — the UI imports `EmotionPredictor`; nothing else needed.
- `reports/confusion_matrix.png`, `training_curves.png`, `class_distribution.png`, `sample_grid.png` — drop into the group report.
- `reports/test_metrics.json` — numbers for the Results section.